# A2A 프로토콜 빈칸 채우기 실습

이 노트북에서는 Ch5에서 배운 **A2A(Agent-to-Agent)** 프로토콜의 핵심 구조를 직접 채워 봅니다.

**실습 방법:**
- 코드 셀의 `___` 부분이 빈칸입니다.
- 힌트를 참고하여 알맞은 코드를 채워 넣으세요.
- 모든 빈칸을 채운 뒤 셀을 실행하면 동작을 확인할 수 있습니다.
- 노트북 맨 하단에 정답이 있습니다.

| 실험 | 내용 | 빈칸 | 시간 |
|------|------|------|------|
| 실험 1 | Agent Card 필수 8개 필드 작성 | 5개 | 5분 |
| 실험 2 | A2A JSON-RPC 메시지 구성 | 4개 | 5분 |
| 실험 3 | A2A 역할 전환 추적 (Client ↔ Remote) | 3개 | 5분 |

In [ ]:
# 공통 설정
import json

print("import 완료")

## 실험 1: Agent Card 필수 필드 작성 (5분)

Agent Card는 A2A 프로토콜에서 Agent의 **자기소개서**입니다.
다른 Agent가 이 Card를 읽고 "이 Agent는 무엇을 할 수 있는가?"를 파악합니다.

아래 빈칸을 채워 Check Agent의 Agent Card를 완성하세요.

**스펙상 필수 8개 필드** ([§4.4.1 AgentCard](https://a2a-protocol.org/latest/specification/#441-agentcard)):

| 필드 | 설명 |
|------|------|
| `name` | Agent 이름 |
| `description` | Agent 설명 |
| `version` | 버전 |
| `supportedInterfaces` | 접속 URL + 프로토콜 바인딩 |
| `capabilities` | 지원 기능 (스트리밍 등) |
| `defaultInputModes` | 입력 MIME 타입 |
| `defaultOutputModes` | 출력 MIME 타입 |
| `skills` | Agent가 가진 능력 목록 |

In [ ]:
# ===== 빈칸을 채우세요 =====

check_agent_card = {
    "name": ___,                    # 힌트: Agent 이름 (문자열)
    "description": "메일함을 확인하고 중요 메일을 감지하여 알림을 요청합니다.",
    "version": "1.0.0",
    "supportedInterfaces": [
        {
            "url": ___,             # 힌트: Check Agent가 실행되는 URL ("http://localhost:9501")
            "protocolBinding": ___, # 힌트: 우리 실습에서 쓰는 바인딩 ("JSONRPC")
            "protocolVersion": "1.0",
        }
    ],
    "capabilities": {"streaming": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {
            "id": ___,             # 힌트: 스킬 식별자 ("check-mail")
            "name": ___,           # 힌트: 스킬 표시 이름 ("메일 확인")
            "description": "메일함을 확인하고 중요 메일을 필터링합니다",
        }
    ],
}

# 검증: 필수 8개 필드 존재 확인
required_fields = [
    "name", "description", "version", "supportedInterfaces",
    "capabilities", "defaultInputModes", "defaultOutputModes", "skills",
]
for field in required_fields:
    assert field in check_agent_card, f"필수 필드 '{field}'가 누락되었습니다"

# 세부 검증
assert check_agent_card["supportedInterfaces"][0]["url"].startswith("http")
assert "9501" in check_agent_card["supportedInterfaces"][0]["url"]
assert check_agent_card["supportedInterfaces"][0]["protocolBinding"] == "JSONRPC"
assert check_agent_card["skills"][0].get("id"), "skills[].id는 필수입니다"

print("Check Agent Card:")
print(json.dumps(check_agent_card, indent=2, ensure_ascii=False))
print()
print(f"필수 8개 필드 모두 존재 확인")
print(f"[A2A Discovery] /.well-known/agent-card.json 으로 이 Card를 발견합니다.")
print()
print("실험 1 통과: Agent Card 구성 완료")

## 실험 2: A2A JSON-RPC 메시지 구성 (5분)

A2A 프로토콜은 **JSON-RPC 2.0** 기반입니다.
Check Agent가 Notify Agent에게 알림을 요청할 때 아래 구조의 메시지를 보냅니다.

빈칸을 채워 A2A 메시지를 완성하세요.

In [ ]:
# ===== 빈칸을 채우세요 =====

# Check Agent → Notify Agent로 보내는 A2A 메시지
a2a_request = {
    "jsonrpc": "2.0",
    "id": "check-to-notify-001",
    "method": ___,           # 힌트: A2A에서 메시지를 보내는 메서드 이름 ("message/send")
    "params": {
        "message": {
            "role": ___,     # 힌트: 요청하는 쪽은 "user", 응답하는 쪽은 "agent" (2가지 중 택 1)
            "parts": [
                {
                    "kind": ___,   # 힌트: 텍스트 메시지의 종류 ("text")
                    "text": ___,   # 힌트: 실제 알림 내용 문자열
                }
            ],
        },
    },
}

# 검증
assert a2a_request["method"] == "message/send", f"A2A 메시지 전송 메서드는 'message/send'입니다"
assert a2a_request["params"]["message"]["role"] == "user"
assert a2a_request["params"]["message"]["parts"][0]["kind"] == "text"
assert len(a2a_request["params"]["message"]["parts"][0]["text"]) > 0

print("A2A JSON-RPC 메시지:")
print(json.dumps(a2a_request, indent=2, ensure_ascii=False))
print()
print("[흐름] Check Agent --POST /--> Notify Agent (localhost:9502)")
print(f"  method: {a2a_request['method']}")
print(f"  role: {a2a_request['params']['message']['role']} (요청 측 = user, 응답 측 = agent)")
print(f"  parts: {a2a_request['params']['message']['parts'][0]['kind']}")
print()
print("실험 2 통과: A2A 메시지 구성 완료")

## 실험 3: A2A 역할 전환 추적 (5분)

우리 실습에서 **Check Agent는 두 가지 역할**을 동시에 합니다:
- 외부 요청을 **받을 때** → Remote Agent (서버)
- Notify Agent에게 알림을 **보낼 때** → Client Agent (클라이언트)

아래 빈칸을 채워 각 구간에서 누가 Client이고 누가 Remote인지 추적하세요.

```
check_and_notify_client.py → Check Agent(:9501) → Notify Agent(:9502)
       (?)                     (? / ?)               (?)
```

In [ ]:
# ===== 빈칸을 채우세요 =====

# 구간 1: check_and_notify_client.py → Check Agent
call_1 = {
    "client_agent": ___,       # 힌트: 요청을 보내는 쪽 ("check_and_notify_client.py")
    "remote_agent": ___,       # 힌트: 요청을 받는 쪽 ("Check Agent")
    "port": 9501,
}

# 구간 2: Check Agent → Notify Agent (중요 메일 발견 시)
call_2 = {
    "client_agent": ___,       # 힌트: 이번에는 Check Agent가 어떤 역할? ("Check Agent")
    "remote_agent": "Notify Agent",
    "port": 9502,
}

# 검증
assert call_1["client_agent"] == "check_and_notify_client.py"
assert call_1["remote_agent"] == "Check Agent"
assert call_2["client_agent"] == "Check Agent"

print("A2A 역할 전환 추적:")
print()
print(f"  구간 1: {call_1['client_agent']}")
print(f"          --[message/send]--> {call_1['remote_agent']} (:{call_1['port']})")
print(f"          Client Agent           Remote Agent")
print()
print(f"  구간 2: {call_2['client_agent']}")
print(f"          --[message/send]--> {call_2['remote_agent']} (:{call_2['port']})")
print(f"          Client Agent           Remote Agent")
print()
print("[핵심] Check Agent는 구간 1에서 Remote, 구간 2에서 Client — 역할이 고정이 아닙니다.")
print("       A2A에서 Client/Remote는 '누가 요청하느냐'에 따라 결정됩니다.")
print()
print("실험 3 통과: A2A 역할 전환 추적 완료")

In [ ]:
# 전체 실험 결과 정리
print("=" * 50)
print("전체 실험 결과")
print("=" * 50)

print("\n[실험 1] Agent Card")
print("  - 필수 8개 필드: name, description, version, supportedInterfaces,")
print("    capabilities, defaultInputModes, defaultOutputModes, skills")
print("  - /.well-known/agent-card.json 으로 다른 Agent가 발견")

print("\n[실험 2] A2A JSON-RPC 메시지")
print("  - method: 'message/send'")
print("  - message.role + message.parts 구조")

print("\n[실험 3] A2A 역할 전환")
print("  - Client Agent: 요청을 보내는 쪽")
print("  - Remote Agent: 요청을 받는 쪽")
print("  - 하나의 Agent(Check Agent)가 상황에 따라 양쪽 역할 수행")

print("\n모든 실험 통과!")

## 핵심 비교

| 구분 | MCP | A2A |
|------|-----|-----|
| **방향** | 수직 (Agent → Tool) | 수평 (Agent ↔ Agent) |
| **단위** | Tool Call (함수 호출) | Task (작업 요청) |
| **발견** | Tool Schema | Agent Card (8개 필수 필드) |
| **프로토콜** | JSON-RPC over stdio/HTTP | JSON-RPC / gRPC / REST (3가지 바인딩) |
| **역할** | 고정 (Host → Server) | 유동적 (같은 Agent가 Client ↔ Remote) |
| **실습 예시** | Ch4: `check_inbox()` 호출 | Ch5: `message/send`로 알림 요청 |

> **우리 실습에서 MCP는 등장하지 않습니다.** Check Agent의 메일 데이터와 Notify Agent의 알림 채널은 A2A 프로토콜 구조에 집중하기 위해 모킹(하드코딩/콘솔 출력)으로 구현했습니다. 프로덕션에서는 Check Agent가 MCP로 메일 서버에, Notify Agent가 MCP로 Slack API에 접근합니다 — 이때도 Agent 간 A2A 통신은 동일합니다.

## 학습 포인트

1. **Agent Card**는 A2A의 발견 메커니즘입니다. 스펙상 **8개 필수 필드**(`name`, `description`, `version`, `supportedInterfaces`, `capabilities`, `defaultInputModes`, `defaultOutputModes`, `skills`)를 갖춰야 합니다.
2. **A2A 메시지**는 JSON-RPC 2.0 기반이며, `message/send` 메서드로 Task를 전달합니다.
3. **Client Agent와 Remote Agent는 고정 역할이 아닙니다.** Check Agent는 외부 요청을 받을 때는 Remote Agent, Notify Agent에게 요청할 때는 Client Agent입니다.
4. **MCP와 A2A는 상호보완적**입니다. 우리 실습은 A2A에 집중하기 위해 MCP를 사용하지 않았지만, 프로덕션에서는 각 Agent 내부에서 MCP로 도구에 접근하고, Agent 간에는 A2A로 협업합니다.

---

## 정답

<details>
<summary><b>클릭하여 정답 확인</b></summary>

### 실험 1: Agent Card

```python
check_agent_card = {
    "name": "Mail Check Agent",
    "description": "메일함을 확인하고 중요 메일을 감지하여 알림을 요청합니다.",
    "version": "1.0.0",
    "supportedInterfaces": [
        {"url": "http://localhost:9501", "protocolBinding": "JSONRPC", "protocolVersion": "1.0"}
    ],
    "capabilities": {"streaming": False},
    "defaultInputModes": ["text/plain"],
    "defaultOutputModes": ["text/plain"],
    "skills": [
        {"id": "check-mail", "name": "메일 확인", "description": "메일함을 확인하고 중요 메일을 필터링합니다"}
    ],
}
```

**해설:** Agent Card의 필수 8개 필드가 모두 포함되어야 합니다. `supportedInterfaces`는 접속 URL과 프로토콜 바인딩(`JSONRPC`/`GRPC`/`HTTP+JSON`)을 명시합니다. `skills[].id`는 스킬 식별자로 필수입니다.

---

### 실험 2: A2A JSON-RPC 메시지

```python
a2a_request = {
    "jsonrpc": "2.0",
    "id": "check-to-notify-001",
    "method": "message/send",
    "params": {
        "message": {
            "role": "user",
            "parts": [{"kind": "text", "text": "중요 메일 3통이 감지되었습니다!"}],
        },
    },
}
```

**해설:**
- `method: "message/send"`: A2A에서 메시지를 보내는 표준 메서드입니다.
- `role: "user"`: 요청을 보내는 측(Client Agent)은 `user` 역할입니다.
- `kind: "text"`: 텍스트 형태의 메시지 파트입니다.
- `text`: 실제 알림 내용을 담습니다.

---

### 실험 3: A2A 역할 전환

```python
call_1 = {
    "client_agent": "check_and_notify_client.py",
    "remote_agent": "Check Agent",
    "port": 9501,
}
call_2 = {
    "client_agent": "Check Agent",
    "remote_agent": "Notify Agent",
    "port": 9502,
}
```

**해설:**
- 구간 1: `check_and_notify_client.py`(Client Agent)가 Check Agent(Remote Agent)에게 `message/send`
- 구간 2: Check Agent가 이번에는 **Client Agent 역할로 전환**하여 Notify Agent(Remote Agent)에게 `message/send`
- 하나의 Agent가 상황에 따라 Client/Remote 양쪽 역할을 수행합니다. A2A에서 역할은 "누가 요청하느냐"로 결정됩니다.

</details>